# Notebook 08 — Final Evaluation


## 1. Load project artifacts

In [1]:

from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "notebooks" / "outputs").exists() and (PROJECT_ROOT.parent / "notebooks" / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_RESULTS_PATH = OUTPUT_DIR / "optimized_baseline_test_results.csv"
THRESHOLD_PATH = OUTPUT_DIR / "validation_threshold_results.csv"
SPLIT_CONFIG_PATH = OUTPUT_DIR / "split_config.json"
FEATURE_REVIEW_PATH = OUTPUT_DIR / "feature_scoring_time_review.csv"
MODEL_METADATA_PATH = MODEL_DIR / "fraud_detection_metadata.json"
MODEL_PATH = MODEL_DIR / "fraud_detection_pipeline.pkl"

required = [
    TEST_RESULTS_PATH, THRESHOLD_PATH, SPLIT_CONFIG_PATH,
    FEATURE_REVIEW_PATH, MODEL_METADATA_PATH, MODEL_PATH
]
missing = [str(p) for p in required if not p.exists()]
print("Project root:", PROJECT_ROOT)
print("Missing required files:", missing)

if missing:
    raise FileNotFoundError(
        "Some required outputs are missing. Run Notebooks 04–07 as needed. "
        f"Missing: {missing}"
    )

test_results = pd.read_csv(TEST_RESULTS_PATH)
threshold_results = pd.read_csv(THRESHOLD_PATH)
feature_review = pd.read_csv(FEATURE_REVIEW_PATH)

with open(SPLIT_CONFIG_PATH, "r", encoding="utf-8") as f:
    split_config = json.load(f)
with open(MODEL_METADATA_PATH, "r", encoding="utf-8") as f:
    model_metadata = json.load(f)

print("\nTest results columns:", test_results.columns.tolist())
print("Threshold report columns:", threshold_results.columns.tolist())
print("Feature review columns:", feature_review.columns.tolist())


Project root: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system
Missing required files: []

Test results columns: ['threshold', 'average_precision', 'precision', 'recall', 'f1', 'f2', 'tn', 'fp', 'fn', 'tp', 'model', 'validation_selected_threshold', 'fit_seconds']
Threshold report columns: ['model', 'threshold', 'average_precision', 'precision', 'recall', 'f1', 'f2', 'tn', 'fp', 'fn', 'tp']
Feature review columns: ['feature', 'expected_source', 'initial_assessment', 'question', 'review_status']


## 2. Review final test metrics

In [2]:

print("TEST RESULTS — all models")
display(test_results)

print("\nVALIDATION THRESHOLD RESULTS")
display(threshold_results)

test_results.to_csv(OUTPUT_DIR / "phase1_final_test_metrics.csv", index=False)
threshold_results.to_csv(OUTPUT_DIR / "phase1_threshold_selection.csv", index=False)
print("\nCopied final evaluation reports into Phase 1 sign-off filenames.")


TEST RESULTS — all models


,threshold,average_precision,precision,recall,f1,f2,tn,fp,fn,tp,model,validation_selected_threshold,fit_seconds
0,0.80,0.956723,0.942170,0.876663,0.908237,0.889025,121837,89,204,1450,Random Forest (balanced),0.80,4.584095
1,0.90,0.639760,0.547606,0.601572,0.573322,0.589944,121104,822,659,995,Logistic Regression (balanced),0.90,1.179516
2,0.05,0.013384,0.000000,0.000000,0.000000,0.000000,121926,0,1654,0,Dummy (most frequent),0.05,0.437553



VALIDATION THRESHOLD RESULTS


,model,threshold,average_precision,precision,recall,f1,f2,tn,fp,fn,tp
0,Random Forest (balanced),0.80,0.938738,0.917431,0.884244,0.900532,0.890688,204837.0,99.0,144.0,1100.0
1,Logistic Regression (balanced),0.90,0.517565,0.363419,0.611736,0.455962,0.538190,203603.0,1333.0,483.0,761.0
2,Dummy (most frequent),0.05,0.006034,0.000000,0.000000,0.000000,0.000000,204936.0,0.0,1244.0,0.0



Copied final evaluation reports into Phase 1 sign-off filenames.


## 3. Confirm the selected threshold is present in the validation report

In [3]:

model_name = model_metadata.get("model_name", "Random Forest (balanced)")
saved_threshold = float(model_metadata["threshold"])

print("Saved model:", model_name)
print("Threshold saved in model metadata:", saved_threshold)

# Try common column names because report schemas may differ by notebook version.
model_col = next((c for c in ["model", "model_name", "Model"] if c in threshold_results.columns), None)
threshold_col = next((c for c in ["threshold", "selected_threshold", "Threshold"] if c in threshold_results.columns), None)

if model_col is None or threshold_col is None:
    print("Could not automatically match model and threshold columns.")
    print("Please inspect the displayed validation threshold report manually.")
    threshold_match = None
else:
    selected_rows = threshold_results.loc[
        threshold_results[model_col].astype(str) == model_name
    ]
    if selected_rows.empty:
        print("Saved model name was not found in threshold report; inspect names manually.")
        threshold_match = False
    else:
        reported_thresholds = pd.to_numeric(selected_rows[threshold_col], errors="coerce").dropna()
        threshold_match = bool(np.any(np.isclose(reported_thresholds, saved_threshold, atol=1e-10, rtol=0)))
        print("Saved threshold appears in validation report:", threshold_match)
        print("Matching report rows:")
        display(selected_rows)


Saved model: Random Forest (balanced)
Threshold saved in model metadata: 0.7999999999999999
Saved threshold appears in validation report: True
Matching report rows:


,model,threshold,average_precision,precision,recall,f1,f2,tn,fp,fn,tp
0,Random Forest (balanced),0.8,0.938738,0.917431,0.884244,0.900532,0.890688,204837.0,99.0,144.0,1100.0


## 4. Review chronological split

In [4]:

print("Split configuration:")
display(pd.DataFrame([split_config]))

cutoff = split_config.get("cutoff_step")
time_col = split_config.get("time_column")
if time_col is None:
    time_col = model_metadata.get("time_column", "step")

print("Chronological cutoff step:", cutoff)
print("Time column:", time_col)
print("Note: Notebook 04's saved report/config is the source of truth for the split.")


Split configuration:


,dataset_path,target_column,time_column,split_method,train_rule,test_rule,cutoff_step,train_step_min,train_step_max,test_step_min,test_step_max,train_rows,test_rows,random_split,resampling_applied,exclude_from_initial_features
0,data/processed/transactions_features.csv.gz,isFraud,step,chronological_by_step,step <= 594,step > 594,594,1,594,595,743,6239040,123580,False,False,"[isFraud, isFlaggedFraud, nameOrig, nameDest]"


Chronological cutoff step: 594
Time column: step
Note: Notebook 04's saved report/config is the source of truth for the split.


## 5. Review the feature scoring-time checklist

In [5]:

print("FEATURE SCORING-TIME REVIEW")
display(feature_review)

status_col = next((c for c in ["review_status", "status", "confirmed_status"] if c in feature_review.columns), None)
if status_col:
    status_text = feature_review[status_col].astype(str).str.upper()
    pending_count = int(status_text.str.contains("PENDING|NOT REVIEWED|UNKNOWN", regex=True).sum())
    print(f"Review rows marked pending/unknown: {pending_count} of {len(feature_review)}")
    feature_review_complete = pending_count == 0
else:
    print("No recognized review-status column found. Manual review is required.")
    feature_review_complete = False

print("Human feature review complete:", feature_review_complete)
print("Important: Do not mark features safe until their availability at scoring time has been confirmed.")


FEATURE SCORING-TIME REVIEW


,feature,expected_source,initial_assessment,question,review_status
0,type,Transaction request,Likely available before decision,Is transaction type present in the authorizati...,PENDING — confirm with project design
1,amount,Transaction request,Likely available before decision,Is requested amount available before authoriza...,PENDING — confirm with project design
2,oldbalanceOrg,Origin account state before transaction,Confirm operational timing,Is the balance snapshot captured before this t...,PENDING — confirm with project design
3,oldbalanceDest,Destination account state before transaction,Confirm operational timing,Is destination balance known and available bef...,PENDING — confirm with project design
4,hour_of_day,Derived from simulation step,Simulation-specific,Can the equivalent time feature be computed at...,PENDING — confirm with project design
5,night_flag,Derived from hour_of_day,Simulation-specific,Is the time zone and night-hour definition spe...,PENDING — confirm with project design
6,amount_to_origin_balance_ratio,Amount divided by old origin balance,Confirm zero handling and timing,Is it calculated only from requested amount an...,PENDING — confirm with project design
7,origin_balance_was_zero,Derived from old origin balance,Likely pre-transaction if based on oldbalanceOrg,"Does this feature use oldbalanceOrg only, not ...",PENDING — confirm with project design
8,origin_txn_count_prior_steps,Historical transactions before current step,Audit grouping and ordering,Does the count exclude the current transaction...,PENDING — confirm with project design


Review rows marked pending/unknown: 9 of 9
Human feature review complete: False
Important: Do not mark features safe until their availability at scoring time has been confirmed.


## 6. Create a final Phase 1 status report

In [7]:

checks = [
    ("Saved model pipeline exists", MODEL_PATH.exists()),
    ("Saved model metadata exists", MODEL_METADATA_PATH.exists()),
    ("Final test metrics report exists", TEST_RESULTS_PATH.exists()),
    ("Validation threshold report exists", THRESHOLD_PATH.exists()),
    ("Feature audit report exists", (OUTPUT_DIR / "feature_quality_audit.csv").exists()),
    ("Scoring-time feature checklist exists", FEATURE_REVIEW_PATH.exists()),
    ("Threshold matches validation report", threshold_match is True),
    ("Human feature review completed", feature_review_complete),
]

status_df = pd.DataFrame(checks, columns=["check", "passed"])
display(status_df)

report_lines = [
    "# Phase 1 — Final Evaluation and Sign-off",
    "",
    f"- Selected model: {model_name}",
    f"- Saved classification threshold: {saved_threshold:.12g}",
    f"- Model pipeline exists: {MODEL_PATH.exists()}",
    f"- Model metadata exists: {MODEL_METADATA_PATH.exists()}",
    f"- Validation threshold matched: {threshold_match}",
    f"- Human feature review completed: {feature_review_complete}",
    "",
    "## Checklist",
]
for name, passed in checks:
    report_lines.append(f"- [{'x' if passed else ' '}] {name}")

report_lines += [
    "",
    "## Required caveats",
    "- Very high test scores require investigation before claiming generalization.",
    "- The scoring-time feature review must confirm each feature is available before a transaction decision.",
    "- Threshold selection must use validation data, not final test labels.",
    "- This report is a project sign-off aid, not a production-readiness certification.",
    ""
]

report_path = OUTPUT_DIR / "phase1_final_signoff.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

status_path = OUTPUT_DIR / "phase1_signoff_checklist.csv"
status_df.to_csv(status_path, index=False)

all_passed = bool(status_df["passed"].all())
print("\nSaved report:", report_path)
print("Saved checklist:", status_path)



,check,passed
0,Saved model pipeline exists,True
1,Saved model metadata exists,True
2,Final test metrics report exists,True
3,Validation threshold report exists,True
4,Feature audit report exists,True
5,Scoring-time feature checklist exists,True
6,Threshold matches validation report,True
7,Human feature review completed,False



Saved report: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\phase1_final_signoff.md
Saved checklist: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\notebooks\outputs\phase1_signoff_checklist.csv
